In [30]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet)
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor)
from xgboost import XGBRegressor as XGBoost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from BimodalPolyRfRegressor import BimodalPolyRfRegressor
from sklearn.model_selection import GridSearchCV
warnings.filterwarnings('ignore')


In [31]:
train_df = pd.read_csv('data/track_k_ecommerce_returns_train.csv')
test_df = pd.read_csv('data/track_k_ecommerce_returns_test.csv')

target = 'return_rate_pct'
X_train = train_df.drop(columns=[target])
y_train = train_df[target]
X_test = test_df.drop(columns=[target])
y_test = test_df[target]

In [32]:
models = {
    'ridge': Ridge(),
    'lasso': Lasso(),
    'elasticnet': ElasticNet(),
    'randomforest': RandomForestRegressor(random_state=42),
    'xgboost': XGBoost(),
    'gradientboost': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'polynomial': Pipeline([
    ('poly', PolynomialFeatures(degree=4)), 
    ('scaler', StandardScaler()),
    ('linear', LinearRegression())]
    ),
    'bimodalpolyrfregressor': BimodalPolyRfRegressor()
}


param_ridge = {"alpha": np.logspace(-2, 4, 20)}
param_lasso = {"alpha": np.logspace(-4, 1, 20)}
param_elasticnet = {"alpha": np.logspace(-4, 1, 20), "l1_ratio": np.linspace(0, 1, 20)}
param_randomforest = {"n_estimators": [50, 100], "max_depth": [None, 5, 10]}

#NOTE: AI GENERATED PARAMS
param_xgboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_gradientboost = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, 7], 
    "learning_rate": [0.01, 0.1, 0.2]
}

param_adaboost = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1, 0.2]
}

param_polynomials = {
    'poly__degree': [i for i in range(2, 5)] # This will test linear (1), quadratic (2), and cubic (3)
}

param_grids = {
    'ridge': param_ridge,
    'lasso': param_lasso,
    'elasticnet': param_elasticnet,
    'randomforest': param_randomforest,
    'xgboost': param_xgboost,
    'gradientboost': param_gradientboost,
    'adaboost': param_adaboost,
    'polynomial': param_polynomials,
    'bimodalpolyrfregressor': {'threshold': [173.96]}
}


In [ ]:
results = []

print("Training\n\n")

for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mse = mean_squared_error(y_test, preds)
        results.append({'Model': name, 'MSE': mse})

    except Exception as e:
        print(f"Failed to train {name}: {e}")

print("\n" + "="*45)
results_df = pd.DataFrame(results).sort_values(by='MSE').reset_index(drop=True)
results_df.index = results_df.index
print(results_df.to_string())

Training



                    Model          MSE
1  bimodalpolyrfregressor    21.883008
2           gradientboost    23.783268
3            randomforest    25.773722
4                 xgboost    27.223045
5                adaboost    31.373914
6                   lasso   122.841120
7              elasticnet   123.029881
8                   ridge   125.962017
9              polynomial  1166.930052


In [34]:
results = []

print("Training\n\n")

for name, model in models.items():
    try:
        grid = grid = GridSearchCV(model, param_grids[name], cv=5)
        grid.fit(X_train, y_train)
        preds = grid.predict(X_test)
        mse = mean_squared_error(y_test, preds)
        results.append({'Model': name, 'MSE': mse})
        print(name, ' Done')

    except Exception as e:
        print(f"Failed to train {name}: {e}")

print("\n" + "="*45)
results_df = pd.DataFrame(results).sort_values(by='MSE').reset_index(drop=True)
results_df.index = results_df.index
print(results_df.to_string())

Training


ridge  Done
lasso  Done
elasticnet  Done
randomforest  Done
xgboost  Done
gradientboost  Done
adaboost  Done
polynomial  Done
bimodalpolyrfregressor  Done

                    Model         MSE
1  bimodalpolyrfregressor   21.883008
2                 xgboost   23.480938
3           gradientboost   23.699062
4            randomforest   25.773722
5                adaboost   32.839521
6              polynomial   45.469941
7                   ridge  124.707438
8              elasticnet  124.873421
9                   lasso  125.467103
